In [ ]:
import sys,json
from pathlib import Path
cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)
print('ready — make sure docker-compose is running: make up')

## 1. pgvector Schema

```sql
CREATE TABLE document_chunks (
    id           UUID PRIMARY KEY,
    content_hash VARCHAR(64) UNIQUE,  -- idempotent upserts
    content      TEXT,
    embedding    vector(1536),          -- HNSW indexed
    country      TEXT,                  -- indexed for filtering
    year         TEXT,
    report_type  TEXT,
    page_number  INTEGER,
    metadata     JSONB
);
```

**Key design decisions:**
- `content_hash` as conflict key → re-ingesting is safe
- Typed columns for filterable fields → fast WHERE clauses
- HNSW index → approximate nearest neighbour, much faster than exact

In [ ]:
from rag.ingestion.indexer import VectorIndex
idx = VectorIndex()
try:
    idx.init_schema()
    print('✅ Schema initialised')
    print(f'Chunks in DB: {idx.count()}')
except Exception as e:
    print(f'❌ DB error: {e}')
    print('Make sure postgres is running: docker-compose up postgres -d')

In [ ]:
import json
from rag.ingestion.loader import load_directory
from rag.ingestion.cleaner import clean_pages
from rag.ingestion.chunker import ChunkStrategy, chunk_pages
data_dir = repo_root/'data'/'raw'
meta_map = json.loads((repo_root/'data'/'metadata.json').read_text())
pages   = load_directory(data_dir, metadata_map=meta_map)
cleaned = clean_pages(pages)
chunks  = chunk_pages(cleaned, ChunkStrategy.RECURSIVE, chunk_size=800)
print(f'Upserting {len(chunks)} chunks...')
idx.init_schema()
n = idx.upsert_documents(chunks[:100])  # small batch first
print(f'Upserted {n} chunks. Total: {idx.count()}')

In [ ]:
from rag.ingestion.embedder import Embedder
from rag.ingestion.indexer import VectorIndex
emb  = Embedder()
idx  = VectorIndex(embedder=emb)
query = 'maternal mortality ratio'
q_vec = emb.embed_query(query)
results = idx.similarity_search(q_vec, top_k=5)
print(f'Query: {query}')
print(f'Top {len(results)} results:')
for r in results:
    print(f"  [{r['score']:.3f}] {r.get('country','')} {r.get('year','')} p{r.get('page_number','')}")
    print(f"    {r['content'][:150]}...")
    print()

In [ ]:
results_filtered = idx.similarity_search(
    q_vec, top_k=5, filters={'country':'Nigeria','year':'2021'})
print(f'Filtered (Nigeria 2021): {len(results_filtered)} results')
for r in results_filtered:
    print(f"  [{r['score']:.3f}] {r['country']} {r['year']} — {r['content'][:100]}...")

## 6. Index type comparison

| | HNSW | IVFFlat |
|---|---|---|
| Build speed | Slower | Faster |
| Query speed | **Faster** | Slower |
| Memory | More | Less |
| Recall | **Higher** | Lower |
| Best for | **Production queries** | Large corpora, memory constrained |

We use HNSW with m=16, ef_construction=64. Good defaults for < 1M vectors.
For our 8,234 chunks, HNSW is instantaneous. At 1M+ chunks, tune ef_search.

## ✅ Episode 4 complete

**Episode 5:** Your first RAG chain — retrieve + generate in 30 lines of LangChain LCEL.